# Problem: Producer-Consumer Problem Using Peterson’s Solution

A producer produces items and places them into a shared buffer.  
A consumer takes items from the shared buffer and consumes them.

The buffer size is limited to 5 items.

The producer must not add items when the buffer is full.  
The consumer must not remove items when the buffer is empty.

# Solve this Producer-Consumer synchronization problem using Peterson’s Solution in Python.

In [ ]:
# Solution to the Producer-Consumer problem using Peterson's Solution for mutual exclusion

import threading
import time
import random

# Buffer size
BUFFER_SIZE = 5

# Shared buffer
buffer = []

# Total items to produce and consume
TOTAL_ITEMS = 10

# Process ID
# Producer = 0
# Consumer = 1

# interested[0] means Producer wants to enter critical section
# interested[1] means Consumer wants to enter critical section
interested = [False, False]

# turn variable decides priority
turn = 0


def Entry_section(process):

    global turn

    # other process calculation
    # If process = 0, other = 1
    # If process = 1, other = 0
    other = 1 - process

    # Current process is interested to enter critical section
    interested[process] = True

    # Current process gives chance to the other process
    turn = process

    # Busy waiting condition
    # If other process is also interested
    # and turn is still current process,
    # then current process will wait
    while interested[other] == True and turn == process:
        time.sleep(0.001)  # small sleep to reduce CPU usage


def Exit_section(process):
    """
    Peterson's Solution Exit Section
    """

    # Current process leaves the critical section
    interested[process] = False


def producer():
    produced = 0

    while produced < TOTAL_ITEMS:

        # Random item produce
        item = random.randint(1, 100)

        while True:
            # Producer enters entry section
            Entry_section(0)

            # Critical Section starts
            # Shared buffer is accessed here
            if len(buffer) < BUFFER_SIZE:
                buffer.append(item)
                produced += 1

                print(f"Producer produced item: {item}")
                print(f"Buffer: {buffer}\n")

                # Producer exits critical section
                Exit_section(0)

                break

            else:
                # If buffer is full, producer cannot add item
                print("Buffer is full. Producer is waiting...\n")

                # Producer exits critical section before waiting
                Exit_section(0)

                time.sleep(1)

        # Remainder section
        time.sleep(1)


def consumer():
    consumed = 0

    while consumed < TOTAL_ITEMS:

        while True:
            # Consumer enters entry section
            Entry_section(1)

            # Critical Section starts
            # Shared buffer is accessed here
            if len(buffer) > 0:
                item = buffer.pop(0)
                consumed += 1

                print(f"Consumer consumed item: {item}")
                print(f"Buffer: {buffer}\n")

                # Consumer exits critical section
                Exit_section(1)

                break

            else:
                # If buffer is empty, consumer cannot consume
                print("Buffer is empty. Consumer is waiting...\n")

                # Consumer exits critical section before waiting
                Exit_section(1)

                time.sleep(1)

        # Remainder section
        time.sleep(2)


# Create producer and consumer threads
producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

# Start both threads
producer_thread.start()
consumer_thread.start()

# Wait for both threads to finish
producer_thread.join()
consumer_thread.join()

print("Producer-Consumer problem solved using Peterson's Solution.")

Producer produced item: 97
Buffer: [97]

Consumer consumed item: 97
Buffer: []

Producer produced item: 21
Buffer: [21]

Producer produced item: 78
Buffer: [21, 78]

Consumer consumed item: 21
Buffer: [78]

Producer produced item: 8
Buffer: [78, 8]

Producer produced item: 23
Buffer: [78, 8, 23]

Consumer consumed item: 78
Buffer: [8, 23]

Producer produced item: 29
Buffer: [8, 23, 29]

Producer produced item: 47
Buffer: [8, 23, 29, 47]

Consumer consumed item: 8
Buffer: [23, 29, 47]

Producer produced item: 47
Buffer: [23, 29, 47, 47]

Consumer consumed item: 23
Buffer: [29, 47, 47]

Producer produced item: 79
Buffer: [29, 47, 47, 79]

Producer produced item: 82
Buffer: [29, 47, 47, 79, 82]

Consumer consumed item: 29
Buffer: [47, 47, 79, 82]

Consumer consumed item: 47
Buffer: [47, 79, 82]

Consumer consumed item: 47
Buffer: [79, 82]

Consumer consumed item: 79
Buffer: [82]

Consumer consumed item: 82
Buffer: []

Producer-Consumer problem solved using Peterson's Solution.


# Solve by MUTEX LOCK

In [ ]:
# Solve the Producer-Consumer problem using simple lock/unlock logic (Mutex Lock)

import threading
import time
import random

# Fixed buffer size
BUFFER_SIZE = 5

# Shared buffer
buffer = []

# Total number of items to produce and consume
TOTAL_ITEMS = 10

# Simple lock variable
# lock = 0 means vacant / free
# lock = 1 means full / busy
lock = 0


def acquire_lock():
    """
    Entry Code:
    This follows the logic shown in the picture.

    while(lock == 1);
    lock = 1;
    """

    global lock

    # If lock is 1, another process/thread is inside critical section
    # So current thread will wait here
    while lock == 1:
        pass

    # When lock becomes 0, current thread takes the lock
    lock = 1


def release_lock():
    """
    Exit Code:
    This follows the logic shown in the picture.

    lock = 0;
    """

    global lock

    # Current thread releases the lock
    lock = 0


def producer():
    produced = 0

    while produced < TOTAL_ITEMS:
        # Producer creates an item
        item = random.randint(1, 100)

        while True:
            # Entry section
            acquire_lock()

            # Critical Section starts
            # Producer accesses shared buffer here
            if len(buffer) < BUFFER_SIZE:
                buffer.append(item)
                produced += 1

                print(f"Producer produced item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit section
                release_lock()

                break

            else:
                # If buffer is full, producer cannot add item
                print("Buffer is full. Producer is waiting...\n")

                # Producer must release lock before waiting
                # Otherwise consumer cannot enter and consume item
                release_lock()

                time.sleep(1)

        # Remainder section
        time.sleep(1)


def consumer():
    consumed = 0

    while consumed < TOTAL_ITEMS:
        while True:
            # Entry section
            acquire_lock()

            # Critical Section starts
            # Consumer accesses shared buffer here
            if len(buffer) > 0:
                item = buffer.pop(0)
                consumed += 1

                print(f"Consumer consumed item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit section
                release_lock()

                break

            else:
                # If buffer is empty, consumer cannot consume item
                print("Buffer is empty. Consumer is waiting...\n")

                # Consumer must release lock before waiting
                # Otherwise producer cannot enter and produce item
                release_lock()

                time.sleep(1)

        # Remainder section
        time.sleep(2)


# Create producer and consumer threads
producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

# Start both threads
producer_thread.start()
consumer_thread.start()

# Wait for both threads to finish
producer_thread.join()
consumer_thread.join()

print("Producer-Consumer problem solved using simple lock/unlock logic.")

Producer produced item: 9
Buffer: [9]

Consumer consumed item: 9
Buffer: []

Producer produced item: 91
Buffer: [91]

Producer produced item: 63
Buffer: [91, 63]

Consumer consumed item: 91
Buffer: [63]

Producer produced item: 55
Buffer: [63, 55]

Consumer consumed item: 63
Buffer: [55]

Producer produced item: 97
Buffer: [55, 97]

Producer produced item: 59
Buffer: [55, 97, 59]

Consumer consumed item: 55
Buffer: [97, 59]

Producer produced item: 43
Buffer: [97, 59, 43]

Producer produced item: 97
Buffer: [97, 59, 43, 97]

Consumer consumed item: 97
Buffer: [59, 43, 97]

Producer produced item: 7
Buffer: [59, 43, 97, 7]

Producer produced item: 78
Buffer: [59, 43, 97, 7, 78]

Consumer consumed item: 59
Buffer: [43, 97, 7, 78]

Consumer consumed item: 43
Buffer: [97, 7, 78]

Consumer consumed item: 97
Buffer: [7, 78]

Consumer consumed item: 7
Buffer: [78]

Consumer consumed item: 78
Buffer: []

Producer-Consumer problem solved using simple lock/unlock logic.


# Solve by Test and  Set instruction

In [5]:
# Solve the Producer-Consumer problem using Test-and-Set instruction

import threading
import time
import random

# Fixed buffer size
BUFFER_SIZE = 5

# Shared buffer between producer and consumer
buffer = []

# Total items producer will produce and consumer will consume
TOTAL_ITEMS = 10

# lock = False means critical section is free
# lock = True means critical section is busy
lock = False

# This internal lock is only used to simulate atomic hardware instruction in Python
# Because real test-and-set is a hardware-level atomic instruction
atomic_guard = threading.Lock()


def test_and_set():
    """
    Test-and-Set logic:

    boolean test_and_set(boolean *target)
    {
        boolean r = *target;
        *target = True;
        return r;
    }

    If lock was False, it returns False and sets lock = True.
    If lock was True, it returns True and process keeps waiting.
    """

    global lock

    # Atomic part starts
    with atomic_guard:
        old_value = lock      # save previous value of lock
        lock = True           # set lock to True
        return old_value      # return previous value


def acquire_lock():
    """
    Entry Section:

    while(test_and_set(&lock));

    If test_and_set() returns True, it means lock is already busy.
    So the process/thread waits.
    """

    while test_and_set() == True:
        time.sleep(0.001)  # small sleep to reduce CPU usage


def release_lock():
    """
    Exit Section:

    lock = false;

    This means current process/thread leaves the critical section.
    """

    global lock

    with atomic_guard:
        lock = False


def producer():
    produced = 0

    while produced < TOTAL_ITEMS:
        # Producer creates a random item
        item = random.randint(1, 100)

        while True:
            # Entry Section using Test-and-Set
            acquire_lock()

            # Critical Section starts
            # Producer accesses shared buffer here
            if len(buffer) < BUFFER_SIZE:
                buffer.append(item)
                produced += 1

                print(f"Producer produced item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit Section
                release_lock()

                break

            else:
                # If buffer is full, producer cannot produce
                print("Buffer is full. Producer is waiting...\n")

                # Producer releases lock so consumer can consume
                release_lock()

                time.sleep(1)

        # Remainder Section
        time.sleep(1)


def consumer():
    consumed = 0

    while consumed < TOTAL_ITEMS:
        while True:
            # Entry Section using Test-and-Set
            acquire_lock()

            # Critical Section starts
            # Consumer accesses shared buffer here
            if len(buffer) > 0:
                item = buffer.pop(0)
                consumed += 1

                print(f"Consumer consumed item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit Section
                release_lock()

                break

            else:
                # If buffer is empty, consumer cannot consume
                print("Buffer is empty. Consumer is waiting...\n")

                # Consumer releases lock so producer can produce
                release_lock()

                time.sleep(1)

        # Remainder Section
        time.sleep(2)


# Create producer and consumer threads
producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

# Start threads
producer_thread.start()
consumer_thread.start()

# Wait for both threads to finish
producer_thread.join()
consumer_thread.join()

print("Producer-Consumer problem solved using Test-and-Set.")

Producer produced item: 27
Buffer: [27]

Consumer consumed item: 27
Buffer: []

Producer produced item: 4
Buffer: [4]

Producer produced item: 4
Buffer: [4, 4]

Consumer consumed item: 4
Buffer: [4]

Producer produced item: 80
Buffer: [4, 80]

Producer produced item: 27
Buffer: [4, 80, 27]

Consumer consumed item: 4
Buffer: [80, 27]

Producer produced item: 21
Buffer: [80, 27, 21]

Producer produced item: 38
Buffer: [80, 27, 21, 38]

Consumer consumed item: 80
Buffer: [27, 21, 38]

Producer produced item: 23
Buffer: [27, 21, 38, 23]

Producer produced item: 21
Buffer: [27, 21, 38, 23, 21]

Consumer consumed item: 27
Buffer: [21, 38, 23, 21]

Producer produced item: 2
Buffer: [21, 38, 23, 21, 2]

Consumer consumed item: 21
Buffer: [38, 23, 21, 2]

Consumer consumed item: 38
Buffer: [23, 21, 2]

Consumer consumed item: 23
Buffer: [21, 2]

Consumer consumed item: 21
Buffer: [2]

Consumer consumed item: 2
Buffer: []

Producer-Consumer problem solved using Test-and-Set.


# Solve By Compare and Swap instruction

In [6]:
# Solve the Producer-Consumer problem using Compare-and-Swap instruction

import threading
import time
import random

# Fixed buffer size
BUFFER_SIZE = 5

# Shared buffer between producer and consumer
buffer = []

# Total items to produce and consume
TOTAL_ITEMS = 10

# lock = 0 means critical section is free
# lock = 1 means critical section is busy
lock = 0

# This lock is used only to simulate atomic compare-and-swap in Python
# Because real Compare-and-Swap is a hardware-level atomic instruction
atomic_guard = threading.Lock()


def compare_and_swap(expected, new_value):
    """
    Compare-and-Swap logic:

    int compare_and_swap(int *value, int expected, int new_value)
    {
        int old = *value;

        if (*value == expected)
            *value = new_value;

        return old;
    }

    Here:
    expected = 0
    new_value = 1

    If lock is 0, it changes lock to 1 and returns 0.
    If lock is already 1, it returns 1 and process waits.
    """

    global lock

    # Atomic section starts
    with atomic_guard:
        old_value = lock

        # If current lock value is equal to expected value,
        # then update lock with new_value
        if lock == expected:
            lock = new_value

        return old_value


def acquire_lock():
    """
    Entry Section:

    while(compare_and_swap(0, 1) != 0);

    If compare_and_swap returns 0, process enters critical section.
    If it returns 1, process waits.
    """

    while compare_and_swap(0, 1) != 0:
        time.sleep(0.001)  # small sleep to reduce CPU usage


def release_lock():
    """
    Exit Section:

    lock = 0;

    This means current process/thread leaves the critical section.
    """

    global lock

    with atomic_guard:
        lock = 0


def producer():
    produced = 0

    while produced < TOTAL_ITEMS:
        # Producer creates a random item
        item = random.randint(1, 100)

        while True:
            # Entry Section using Compare-and-Swap
            acquire_lock()

            # Critical Section starts
            # Producer accesses shared buffer here
            if len(buffer) < BUFFER_SIZE:
                buffer.append(item)
                produced += 1

                print(f"Producer produced item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit Section
                release_lock()

                break

            else:
                # If buffer is full, producer cannot produce
                print("Buffer is full. Producer is waiting...\n")

                # Producer releases lock so consumer can consume
                release_lock()

                time.sleep(1)

        # Remainder Section
        time.sleep(1)


def consumer():
    consumed = 0

    while consumed < TOTAL_ITEMS:
        while True:
            # Entry Section using Compare-and-Swap
            acquire_lock()

            # Critical Section starts
            # Consumer accesses shared buffer here
            if len(buffer) > 0:
                item = buffer.pop(0)
                consumed += 1

                print(f"Consumer consumed item: {item}")
                print(f"Buffer: {buffer}\n")

                # Exit Section
                release_lock()

                break

            else:
                # If buffer is empty, consumer cannot consume
                print("Buffer is empty. Consumer is waiting...\n")

                # Consumer releases lock so producer can produce
                release_lock()

                time.sleep(1)

        # Remainder Section
        time.sleep(2)


# Create producer and consumer threads
producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

# Start both threads
producer_thread.start()
consumer_thread.start()

# Wait for both threads to finish
producer_thread.join()
consumer_thread.join()

print("Producer-Consumer problem solved using Compare-and-Swap.")

Producer produced item: 71
Buffer: [71]

Consumer consumed item: 71
Buffer: []

Producer produced item: 2
Buffer: [2]

Producer produced item: 55
Buffer: [2, 55]

Consumer consumed item: 2
Buffer: [55]

Producer produced item: 65
Buffer: [55, 65]

Producer produced item: 26
Buffer: [55, 65, 26]

Consumer consumed item: 55
Buffer: [65, 26]

Producer produced item: 98
Buffer: [65, 26, 98]

Producer produced item: 85
Buffer: [65, 26, 98, 85]

Consumer consumed item: 65
Buffer: [26, 98, 85]

Producer produced item: 96
Buffer: [26, 98, 85, 96]

Producer produced item: 86
Buffer: [26, 98, 85, 96, 86]

Consumer consumed item: 26
Buffer: [98, 85, 96, 86]

Producer produced item: 24
Buffer: [98, 85, 96, 86, 24]

Consumer consumed item: 98
Buffer: [85, 96, 86, 24]

Consumer consumed item: 85
Buffer: [96, 86, 24]

Consumer consumed item: 96
Buffer: [86, 24]

Consumer consumed item: 86
Buffer: [24]

Consumer consumed item: 24
Buffer: []

Producer-Consumer problem solved using Compare-and-Swap.


# Solve by Semaphore

In [7]:
import threading
import time
import random

# ---------------------------------------------
# Producer-Consumer using Semaphore
# Following the exact logic of the given picture
# ---------------------------------------------

# Buffer size N = 6 (same as picture)
N = 6

# Shared circular buffer
buffer = [None] * N

# in_index = position where Producer will insert item
# out_index = position where Consumer will remove item
in_index = 0
out_index = 0

# ---------------------------------------------
# Semaphores
# ---------------------------------------------

# empty = number of empty slots in buffer
# Initially all 6 slots are empty
empty = threading.Semaphore(N)

# full = number of filled slots in buffer
# Initially no slot is filled
full = threading.Semaphore(0)

# s = binary semaphore / mutex semaphore
# s = 1 means CS is free
# s = 0 means CS is busy
s = threading.Semaphore(1)

# Total items to produce and consume
TOTAL_ITEMS = 10


def producer():
    global in_index

    for i in range(TOTAL_ITEMS):
        # Producer creates an item
        item = random.randint(1, 100)

        # ---------------- Entry Section ----------------
        # down(empty) -> wait if no empty slot available
        empty.acquire()

        # down(s) -> enter critical section
        s.acquire()
        # ------------------------------------------------

        # ---------------- Critical Section ----------------
        # Place item into buffer[in]
        buffer[in_index] = item
        print(f"Producer produced: {item} at position {in_index}")

        # Move in pointer circularly
        in_index = (in_index + 1) % N

        print("Buffer:", buffer)
        print("-" * 50)
        # --------------------------------------------------

        # ---------------- Exit Section ----------------
        # up(s) -> leave critical section
        s.release()

        # up(full) -> one more slot is now full
        full.release()
        # ----------------------------------------------

        time.sleep(1)


def consumer():
    global out_index

    for i in range(TOTAL_ITEMS):

        # ---------------- Entry Section ----------------
        # down(full) -> wait if no filled slot available
        full.acquire()

        # down(s) -> enter critical section
        s.acquire()
        # ------------------------------------------------

        # ---------------- Critical Section ----------------
        # Take item from buffer[out]
        item = buffer[out_index]
        buffer[out_index] = None
        print(f"Consumer consumed: {item} from position {out_index}")

        # Move out pointer circularly
        out_index = (out_index + 1) % N

        print("Buffer:", buffer)
        print("-" * 50)
        # --------------------------------------------------

        # ---------------- Exit Section ----------------
        # up(s) -> leave critical section
        s.release()

        # up(empty) -> one more slot is now empty
        empty.release()
        # ----------------------------------------------

        time.sleep(2)


# Create producer and consumer threads
producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

# Start execution
producer_thread.start()
consumer_thread.start()

# Wait until both finish
producer_thread.join()
consumer_thread.join()

print("Producer-Consumer problem solved using Semaphore.")

Producer produced: 1 at position 0
Buffer: [1, None, None, None, None, None]
--------------------------------------------------
Consumer consumed: 1 from position 0
Buffer: [None, None, None, None, None, None]
--------------------------------------------------
Producer produced: 74 at position 1
Buffer: [None, 74, None, None, None, None]
--------------------------------------------------
Producer produced: 32 at position 2
Buffer: [None, 74, 32, None, None, None]
--------------------------------------------------
Consumer consumed: 74 from position 1
Buffer: [None, None, 32, None, None, None]
--------------------------------------------------
Producer produced: 43 at position 3
Buffer: [None, None, 32, 43, None, None]
--------------------------------------------------
Producer produced: 47 at position 4
Buffer: [None, None, 32, 43, 47, None]
--------------------------------------------------
Consumer consumed: 32 from position 2
Buffer: [None, None, None, 43, 47, None]
----------------